In [ ]:
from keras.layers import BatchNormalization
from keras.preprocessing.image import load_img,img_to_array
from sklearn.metrics import mean_squared_error
from keras.initializers import RandomNormal
from keras.applications.vgg16 import VGG16
from keras.optimizers import SGD
from keras.models import Model,Sequential
from keras.layers import *
from keras import backend as K
from keras.models import model_from_json
from matplotlib import cm as CM
import matplotlib.pyplot as plt
import tensorflow as tf
from tqdm import tqdm
import scipy.io as io
from PIL import Image
import PIL
import h5py
import os
import glob
import cv2
import random
import math
import sys

In [5]:
K.clear_session()
root = os.path.join(os.getcwd(),'data')
print(root)

c:\Users\User\OneDrive\Desktop\crowd_counting_project\data


In [6]:
part_A_train = os.path.join(root,'part_A_final/train_data','images')
part_A_test = os.path.join(root,'part_A_final/test_data','images')
part_B_train = os.path.join(root,'part_B_final/train_data','images')
part_B_test = os.path.join(root,'part_B_final/test_data','images')
temp = 'test_images'
path_sets = [part_A_train]

In [84]:
img_paths = []

for path in path_sets:
    
    for img_path in glob.glob(os.path.join(path, '*.jpg')):
        
        img_paths.append(str(img_path))
        
print("Total images : ",len(img_paths))

Total images :  300


In [83]:
import numpy as np


def create_img(path):
    #Function to load,normalize and return image 
    im = Image.open(path).convert('RGB')
    
    # Resize image to model input size (224x224)
    im = im.resize((224, 224), Image.Resampling.LANCZOS)
    im = np.array(im)
    
    im = im/255.0
    
    im[:,:,0]=(im[:,:,0]-0.485)/0.229
    im[:,:,1]=(im[:,:,1]-0.456)/0.224
    im[:,:,2]=(im[:,:,2]-0.406)/0.225

    #print(im.shape)
    #im = np.expand_dims(im,axis  = 0)
    return im

def get_input(path):
    img = create_img(path)
    return(img)
    
    
    
def get_output(path):
    #import target
    #resize target
    
    gt_file = h5py.File(path,'r')
    
    target = np.asarray(gt_file['density'])
    
    # Resize target to match the model's expected output size
    # Since model input is 224x224 and we have 3 pooling layers (each divides by 2)
    # Output should be 224/8 = 28x28
    target_resized = cv2.resize(target, (28, 28), interpolation=cv2.INTER_CUBIC)
    
    # Scale by 64 to maintain density values
    target_resized = target_resized * 64
    
    img = np.expand_dims(target_resized, axis=-1)
    
    #print(img.shape)
    
    return img
    
    
    
def preprocess_input(image,target):
    #crop image
    #crop target
    #resize target
    crop_size = (int(image.shape[0]/2),int(image.shape[1]/2))
    
    
    if random.randint(0,9)<= -1:            
            dx = int(random.randint(0,1)*image.shape[0]*1./2)
            dy = int(random.randint(0,1)*image.shape[1]*1./2)
    else:
            dx = int(random.random()*image.shape[0]*1./2)
            dy = int(random.random()*image.shape[1]*1./2)

    #print(crop_size , dx , dy)
    img = image[dx : crop_size[0]+dx , dy:crop_size[1]+dy]
    
    target_aug = target[dx:crop_size[0]+dx,dy:crop_size[1]+dy]
    #print(img.shape)

    return(img,target_aug)

In [82]:
# def image_generator(files, batch_size=64):
#     while True:
#         batch_paths = np.random.choice(files, size=batch_size)
        
#         batch_input = []
#         batch_output = []
        
#         for path in batch_paths:
#             inputt = get_input(path)  # process single image
#             output = get_output(path.replace('.jpg', '.h5').replace('images', 'ground_truth'))
            
#             batch_input.append(inputt)
#             batch_output.append(output)
        
#         batch_x = np.array(batch_input)
#         batch_y = np.array(batch_output)
        
#         yield (batch_x, batch_y)

def image_generator(files, batch_size=4):
    while True:
        if len(files) == 0:
            raise ValueError("No image files found. Check your dataset path.")
        
        batch_input, batch_output = [], []
        batch_paths = random.sample(files, batch_size) if len(files) >= batch_size else files

        for path in batch_paths:
            gt_path = path.replace('.jpg', '.h5').replace('images', 'ground_truth')
            
            if not os.path.exists(gt_path):
                print(f"⚠️ Skipping {path}: ground truth file not found.")
                continue
            
            try:
                inputt = get_input(path)
                output = get_output(gt_path)
                batch_input.append(inputt)
                batch_output.append(output)
            except Exception as e:
                print(f"⚠️ Skipping {path}: {e}")
                continue

        if len(batch_input) == 0:
            continue  # skip empty batch

        yield (np.array(batch_input), np.array(batch_output))



In [81]:
gen = image_generator(img_paths, 1)
x, y = next(gen)
print("Input shape:", x.shape, "Output shape:", y.shape)
print("Expected input: (batch, 224, 224, 3)")
print("Expected output: (batch, 28, 28, 1)")


Input shape: (1, 427, 640, 3) Output shape: (1, 53, 80, 1)
Expected input: (batch, 224, 224, 3)
Expected output: (batch, 28, 28, 1)


In [100]:
# def save_mod(model , str1 , str2):
#     model.save_weights(str1)
    
#     model_json = model.to_json()
    
#     with open(str2, "w") as json_file:
#         json_file.write(model_json)

import os

def save_mod(model, weights_path, json_path):
    # Make sure directories exist
    os.makedirs(os.path.dirname(weights_path), exist_ok=True)
    os.makedirs(os.path.dirname(json_path), exist_ok=True)
    
    # Save model weights
    model.save_weights(weights_path)  # must end with .weights.h5
    
    # Save model architecture as JSON
    model_json = model.to_json()
    with open(json_path, "w") as json_file:
        json_file.write(model_json)



In [86]:
def init_weights_vgg(model):
    #vgg =  VGG16(weights='imagenet', include_top=False)
    
    json_file = open('models/VGG_16.json', 'r')
    loaded_model_json = json_file.read()
    json_file.close()
    loaded_model = model_from_json(loaded_model_json)
    loaded_model.load_weights("weights/VGG_16.h5")
    
    vgg = loaded_model
    
    vgg_weights=[]                         
    for layer in vgg.layers:
        if('conv' in layer.name):
            vgg_weights.append(layer.get_weights())
    
    
    offset=0
    i=0
    while(i<10):
        if('conv' in model.layers[i+offset].name):
            model.layers[i+offset].set_weights(vgg_weights[i])
            i=i+1
            #print('h')
            
        else:
            offset=offset+1

    return (model)

In [87]:
import tensorflow as tf

def euclidean_distance_loss(y_true, y_pred):
    min_h = tf.minimum(tf.shape(y_true)[1], tf.shape(y_pred)[1])
    min_w = tf.minimum(tf.shape(y_true)[2], tf.shape(y_pred)[2])
    
    y_true = y_true[:, :min_h, :min_w, :]
    y_pred = y_pred[:, :min_h, :min_w, :]
    
    return tf.sqrt(tf.reduce_sum(tf.square(y_pred - y_true), axis=[1,2,3]))



In [88]:
# Neural network model : VGG + Conv
def CrowdNet():  
            #Variable Input Size
            rows = None
            cols = None
            
            #Batch Normalisation option
            
            batch_norm = 0
            kernel = (3, 3)
            init = RandomNormal(stddev=0.01)
            model = Sequential() 

            model.add(Input(shape=(224,224,3)))
            model.add(Conv2D(32, (3,3), activation='relu'))
            
            #custom VGG:
            
            if(batch_norm):
                model.add(Conv2D(64, kernel_size = kernel, input_shape = (rows,cols,3),activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(Conv2D(64, kernel_size = kernel,activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='same'))
                model.add(Conv2D(128,kernel_size = kernel, activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(Conv2D(128,kernel_size = kernel, activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='same'))
                model.add(Conv2D(256,kernel_size = kernel, activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(Conv2D(256,kernel_size = kernel, activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(Conv2D(256,kernel_size = kernel, activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='same'))            
                model.add(Conv2D(512, kernel_size = kernel,activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(Conv2D(512, kernel_size = kernel,activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                model.add(Conv2D(512, kernel_size = kernel,activation = 'relu', padding='same'))
                model.add(BatchNormalization())
                
            else:
                model.add(Conv2D(64, kernel_size = kernel,activation = 'relu', padding='same',input_shape = (rows, cols, 3), kernel_initializer = init))
                model.add(Conv2D(64, kernel_size = kernel,activation = 'relu', padding='same', kernel_initializer = init))
                model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='same'))
                model.add(Conv2D(128,kernel_size = kernel, activation = 'relu', padding='same', kernel_initializer = init))
                model.add(Conv2D(128,kernel_size = kernel, activation = 'relu', padding='same', kernel_initializer = init))
                model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='same'))
                model.add(Conv2D(256,kernel_size = kernel, activation = 'relu', padding='same', kernel_initializer = init))
                model.add(Conv2D(256,kernel_size = kernel, activation = 'relu', padding='same', kernel_initializer = init))
                model.add(Conv2D(256,kernel_size = kernel, activation = 'relu', padding='same', kernel_initializer = init))
                model.add(MaxPooling2D(pool_size=(2,2), strides=2, padding='same'))            
                model.add(Conv2D(512, kernel_size = kernel,activation = 'relu', padding='same', kernel_initializer = init))
                model.add(Conv2D(512, kernel_size = kernel,activation = 'relu', padding='same', kernel_initializer = init))
                model.add(Conv2D(512, kernel_size = kernel,activation = 'relu', padding='same', kernel_initializer = init))
                
                

                
            #Conv2D
            model.add(Conv2D(512, (3, 3), activation='relu', dilation_rate = 2, kernel_initializer = init, padding = 'same'))
            model.add(Conv2D(512, (3, 3), activation='relu', dilation_rate = 2, kernel_initializer = init, padding = 'same'))
            model.add(Conv2D(512, (3, 3), activation='relu', dilation_rate = 2, kernel_initializer = init, padding = 'same'))
            model.add(Conv2D(256, (3, 3), activation='relu', dilation_rate = 2, kernel_initializer = init, padding = 'same'))
            model.add(Conv2D(128, (3, 3), activation='relu', dilation_rate = 2, kernel_initializer = init, padding = 'same'))
            model.add(Conv2D(64, (3, 3), activation='relu', dilation_rate = 2, kernel_initializer = init, padding = 'same'))
            model.add(Conv2D(1, (1, 1), activation='relu', dilation_rate = 1, kernel_initializer = init, padding = 'same'))
        
            sgd = SGD(learning_rate = 1e-7, momentum = 0.95)
            model.compile(optimizer=sgd, loss=euclidean_distance_loss, metrics=['mse'])
            
            # model = init_weights_vgg(model)
            
            return model

In [89]:
model = CrowdNet()

In [90]:
model.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_139 (Conv2D)             │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_140 (Conv2D)             │ (None, 222, 222, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_141 (Conv2D)             │ (None, 222, 222, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_24 (MaxPooling2D) │ (None, 111, 111, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_142 (Conv2D)             │ (None, 111, 111, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_143 (Conv2D)             │ (None, 111, 111, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_25 (MaxPooling2D) │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_144 (Conv2D)             │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_145 (Conv2D)             │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_146 (Conv2D)             │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_26 (MaxPooling2D) │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_147 (Conv2D)             │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_148 (Conv2D)             │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_149 (Conv2D)             │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_150 (Conv2D)             │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_151 (Conv2D)             │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_152 (Conv2D)             │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_153 (Conv2D)             │ (None, 28, 28, 256)    │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_154 (Conv2D)             │ (None, 28, 28, 128)    │       295,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_155 (Conv2D)             │ (None, 28, 28, 64)     │        73,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_156 (Conv2D)             │ (None, 28, 28, 1)      │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,281,089 (62.11 MB)

 Trainable params: 16,281,089 (62.11 MB)

 Non-trainable params: 0 (0.00 B)

In [91]:
train_gen = image_generator(img_paths,1)

In [92]:
sgd = SGD(learning_rate = 1e-7, momentum = 0.95)
model.compile(optimizer=sgd, loss=euclidean_distance_loss, metrics=['mse'])

In [93]:
# Test with a smaller number of steps first to verify the fix
model.fit(train_gen, epochs=1, steps_per_epoch=10, verbose=1)


10/10 ━━━━━━━━━━━━━━━━━━━━ 12s 537ms/step - loss: 3.1111 - mse: 0.0199


In [95]:
model.fit(train_gen,epochs=15,steps_per_epoch= 700 , verbose=1)

Epoch 1/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 425s 606ms/step - loss: 4.6766 - mse: 0.0528
Epoch 2/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 485s 693ms/step - loss: 4.2487 - mse: 0.0433
Epoch 3/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 453s 647ms/step - loss: 4.1986 - mse: 0.0390
Epoch 4/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 448s 640ms/step - loss: 4.4871 - mse: 0.0485
Epoch 5/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 442s 632ms/step - loss: 4.6508 - mse: 0.0497
Epoch 6/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 441s 630ms/step - loss: 4.3973 - mse: 0.0444
Epoch 7/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 445s 635ms/step - loss: 4.3386 - mse: 0.0432
Epoch 8/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 442s 631ms/step - loss: 4.9106 - mse: 0.0576
Epoch 9/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 443s 633ms/step - loss: 4.4536 - mse: 0.0475
Epoch 10/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 448s 640ms/step - loss: 4.3446 - mse: 0.0464
Epoch 11/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 459s 655ms/step - loss: 4.4860 - mse: 0.0462
Epoch 12/15
700/700 ━━━━━━━━━━━━━━━━━━━━ 461s 658ms/step - loss

In [101]:
save_mod(model, "weights/model_A_weights.weights.h5", "models/Model.json")
